# 02 — Oracle compression (feasibility test 1)

Can a strong LLM compress Hebrew **by deletion only** while preserving meaning?
`gemini-3.8-flash`, the strongest Gemini model free on our key, deletes ~30% and ~50% of the
words of 20 FLORES+ paragraphs; we measure the token savings and check with question answering
(by `gemini-3.5-flash-lite`) whether the facts survive.

## 1. Imports

Import every library the notebook uses, including the shared `hebrew_token_tax` package.

In [ ]:
import hashlib
import json
import os
import time
import unicodedata
from collections import Counter
from collections.abc import Callable
from functools import partial
from pathlib import Path
from typing import Annotated, Literal

import httpx
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types
from pydantic import BaseModel, Field

from hebrew_token_tax.counters import (
    make_gemini_counter,
    make_openai_counter,
    measure_overhead,
    raw_gemini_count,
    with_backoff,
)
from hebrew_token_tax.data import build_paragraphs, load_aligned_pairs

## 2. Environment & client

Load `.env`, check the keys, and create one Gemini client per API key (free tier only).
Calls start on key 1 and move to key 2 when key 1 hits a daily quota. Keys are never printed.

In [ ]:
GEMINI_KEY_NAMES = ("GEMINI_API_KEY", "GEMINI_API_KEY_2")
REQUIRED_KEYS = ("HF_TOKEN", *GEMINI_KEY_NAMES)

load_dotenv()
missing = [key for key in REQUIRED_KEYS if not os.getenv(key)]
if missing:
    raise RuntimeError(f"Missing from .env: {', '.join(missing)}")

gemini_clients = [genai.Client(api_key=os.environ[name]) for name in GEMINI_KEY_NAMES]
key_index = 0
gemini_client = gemini_clients[key_index]


def switch_to_next_key() -> bool:
    """Move all later calls to the next API key; return False if none is left."""
    global key_index, gemini_client
    if key_index + 1 >= len(gemini_clients):
        return False
    key_index += 1
    gemini_client = gemini_clients[key_index]
    print(f"daily quota reached; switching to key {key_index + 1}")
    return True

## 3. Settings, cache & oracle calls

The compressor is the strongest model that generates for free (AI Studio: `gemini-4-argon` and
`gemini-3.1-pro-preview` have a free limit of 0). Questions, answers and judging use
`gemini-3.5-flash-lite` (500 requests/day) to save the compressor's 20 requests/day.
The pause after each call follows the model's free-tier requests per minute, and a call that
gets no answer within 300 s stops the cell with a timeout error instead of hanging.
Calls to `gemini-3.5-flash-lite` are retried once after a 504; `gemini-3.8-flash` is not.
A per-minute 429 is retried with backoff; a daily-quota 429 switches to the next key at once.
Every API response is cached as JSON in `data/cache/02/`, so a rerun makes no new calls.

`RUN_SUFFIX` picks the run: `""` compresses with `gemini-3.8-flash`; `"_lite"` compresses with
`gemini-3.5-flash-lite`, with its own cache (`data/cache/02_lite/`) and results files (`*_lite`).

In [ ]:
COMPRESSOR_MODELS = {"": "gemini-3.8-flash", "_lite": "gemini-3.5-flash-lite"}
RUN_SUFFIX = ""
COMPRESSOR_MODEL = COMPRESSOR_MODELS[RUN_SUFFIX]
QUESTION_MODEL = "gemini-3.5-flash-lite"
ANSWERER_MODEL = "gemini-3.5-flash-lite"
JUDGE_MODEL = "gemini-3.5-flash-lite"
GEMINI_TOKEN_COUNT_MODELS = {
    "gemini_3_8_flash": "gemini-3.8-flash",
    "gemini_4_argon": "gemini-4-argon",
}
FREE_TIER_RPM = {"gemini-3.8-flash": 5, "gemini-3.5-flash-lite": 15}
RETRY_504_MODELS = frozenset({"gemini-3.5-flash-lite"})
GATEWAY_TIMEOUT_CODE = 504
RATE_LIMIT_CODE = 429
DAILY_QUOTA_MARKER = "perday"
RUN_FREE_TIER_PROBE = False
FREE_TIER_CANDIDATES = (
    "gemini-4-argon",
    "gemini-3.1-pro-preview",
    "gemini-3.8-flash",
    "gemini-3.5-flash-lite",
)

TEMPERATURE = 0.0
TARGETS = {"30%": 0.30, "50%": 0.50}
N_SOURCE_PARAGRAPHS = 50
N_PARAGRAPHS = 20
SEED = 42
N_QUESTIONS = 3
VERDICTS = ("match", "partial", "mismatch")
NOT_IN_TEXT = "לא מופיע בטקסט"
RESULTS_DIR = Path("..") / "results"
RESULTS_CSV = RESULTS_DIR / f"02_oracle_compression{RUN_SUFFIX}.csv"
RESULTS_PNG = RESULTS_DIR / f"02_oracle_compression{RUN_SUFFIX}.png"
CACHE_DIR = Path("..") / "data" / "cache" / f"02{RUN_SUFFIX}"
CACHE_KEY_CHARS = 16
SECONDS_PER_MINUTE = 60
PAUSE_MARGIN_SECONDS = 1.0
GENERATE_BACKOFF_SECONDS = 5.0
GENERATE_TIMEOUT_SECONDS = 300
MILLISECONDS_PER_SECOND = 1000
COUNT_PAUSE_SECONDS = 0.5
N_COMPRESSION_CALLS = N_PARAGRAPHS * len(TARGETS)


def generate_pause_seconds(model: str) -> float:
    """Seconds to wait after a call so `model` stays under its free requests per minute."""
    return SECONDS_PER_MINUTE / FREE_TIER_RPM[model] + PAUSE_MARGIN_SECONDS


for role, model in (
    ("compressor", COMPRESSOR_MODEL),
    ("questions", QUESTION_MODEL),
    ("answerer", ANSWERER_MODEL),
    ("judge", JUDGE_MODEL),
):
    print(f"{role + ':':12} {model} (pause {generate_pause_seconds(model):.0f}s)")
print(f"{'tokenizers:':12} o200k_base, {', '.join(GEMINI_TOKEN_COUNT_MODELS.values())}")
print(f"{'cache:':12} {CACHE_DIR}")

if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(f"{RESULTS_DIR} not found; run from the notebooks/ folder")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

api_calls: Counter[str] = Counter()
http_options = types.HttpOptions(timeout=GENERATE_TIMEOUT_SECONDS * MILLISECONDS_PER_SECOND)


class Compression(BaseModel):
    compressed: str


class Questions(BaseModel):
    questions: Annotated[list[str], Field(min_length=N_QUESTIONS, max_length=N_QUESTIONS)]


class Answers(BaseModel):
    answers: Annotated[list[str], Field(min_length=N_QUESTIONS, max_length=N_QUESTIONS)]


class Verdict(BaseModel):
    verdict: Literal["match", "partial", "mismatch"]
    reason: str


class Judgement(BaseModel):
    verdicts: Annotated[list[Verdict], Field(min_length=N_QUESTIONS, max_length=N_QUESTIONS)]


def cached_call(
    kind: str, request: dict[str, object], call: Callable[[], object], pause: float
) -> object:
    """Return the cached response to `request`, or call the API, cache the result and pause."""
    payload = json.dumps({"kind": kind, **request}, ensure_ascii=False, sort_keys=True)
    key = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:CACHE_KEY_CHARS]
    path = CACHE_DIR / f"{kind}_{key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))["response"]
    response = call()
    record = {"kind": kind, "request": request, "response": response}
    path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    api_calls[kind] += 1
    time.sleep(pause)
    return response


def is_daily_quota(error: errors.APIError) -> bool:
    """True if `error` is a 429 whose details name a per-day quota."""
    details = json.dumps(error.details, default=str).lower()
    return error.code == RATE_LIMIT_CODE and DAILY_QUOTA_MARKER in details


def generate_on_any_key(
    model: str, prompt: str, config: types.GenerateContentConfig
) -> types.GenerateContentResponse:
    """Generate with the current key, moving to the next key when its daily quota is reached."""
    while True:
        try:
            return with_backoff(
                lambda: gemini_client.models.generate_content(
                    model=model, contents=prompt, config=config
                ),
                GENERATE_BACKOFF_SECONDS,
                give_up=is_daily_quota,
            )
        except errors.APIError as error:
            if not is_daily_quota(error) or not switch_to_next_key():
                raise


def retry_once_on_gateway_timeout[T](model: str, call: Callable[[], T]) -> T:
    """Run `call`; for models in RETRY_504_MODELS, retry once after a 504."""
    try:
        return call()
    except errors.APIError as error:
        if error.code != GATEWAY_TIMEOUT_CODE or model not in RETRY_504_MODELS:
            raise
        time.sleep(GENERATE_BACKOFF_SECONDS)
        return call()


def call_oracle(model: str, prompt: str, schema: type[BaseModel]) -> str:
    """Ask `model` at temperature 0 for JSON matching `schema`, giving up after the timeout."""
    config = types.GenerateContentConfig(
        temperature=TEMPERATURE,
        response_mime_type="application/json",
        response_schema=schema,
        http_options=http_options,
    )
    try:
        response = retry_once_on_gateway_timeout(
            model, partial(generate_on_any_key, model, prompt, config)
        )
    except httpx.TimeoutException as error:
        raise TimeoutError(
            f"{model} did not answer within {GENERATE_TIMEOUT_SECONDS}s; "
            "rerun the cell (finished calls are cached)"
        ) from error
    if response.text is None:
        raise RuntimeError(f"Empty response from {model}: {response.prompt_feedback}")
    return response.text


def generate_json[Schema: BaseModel](
    kind: str, model: str, prompt: str, schema: type[Schema]
) -> Schema:
    """Return `model`'s parsed answer to `prompt`, from the cache when possible."""
    request = {
        "model": model,
        "temperature": TEMPERATURE,
        "schema": schema.__name__,
        "prompt": prompt,
    }
    text = cached_call(
        kind,
        request,
        partial(call_oracle, model, prompt, schema),
        generate_pause_seconds(model),
    )
    return schema.model_validate_json(text)

## 4. Free-tier probe

Opt-in (`RUN_FREE_TIER_PROBE` in cell 3), because each probe spends one request of each
model's daily quota. One tiny `generate_content` call per candidate (strongest first) shows which
models generate for free on this key, plus any rate limits the API exposes. Stops the notebook if
`COMPRESSOR_MODEL` is not the strongest free model, or if its known daily limit is too low.

In [ ]:
PROBE_PROMPT = "Reply with OK"
PROBE_MAX_OUTPUT_TOKENS = 5
QUOTA_FAILURE_TYPE = "type.googleapis.com/google.rpc.QuotaFailure"
LIMIT_NAME_PARTS = ("ratelimit", "quota")
DAILY_NAME_PART = "day"
ERROR_MESSAGE_CHARS = 200


def quota_limits(error: errors.APIError) -> dict[str, str]:
    """Map each quota ID named in an error (e.g. '...PerDay...-FreeTier') to its limit."""
    body = error.details if isinstance(error.details, dict) else {}
    return {
        violation["quotaId"]: str(violation["quotaValue"])
        for detail in body.get("error", {}).get("details", [])
        if detail.get("@type") == QUOTA_FAILURE_TYPE
        for violation in detail.get("violations", [])
        if "quotaId" in violation and "quotaValue" in violation
    }


def header_limits(response: types.GenerateContentResponse) -> dict[str, str]:
    """Return any rate-limit or quota headers of a successful response."""
    http_response = response.sdk_http_response
    headers = (http_response.headers or {}) if http_response is not None else {}
    return {
        name: value
        for name, value in headers.items()
        if any(part in name.lower() for part in LIMIT_NAME_PARTS)
    }


def probe_free_generation(model: str) -> dict[str, object]:
    """Send one tiny generate_content call to `model`; report if it works and any limits."""
    config = types.GenerateContentConfig(
        max_output_tokens=PROBE_MAX_OUTPUT_TOKENS, http_options=http_options
    )
    try:
        response = gemini_client.models.generate_content(
            model=model, contents=PROBE_PROMPT, config=config
        )
    except errors.APIError as error:
        reason = f"{error.code} {error.status}: {error.message}"[:ERROR_MESSAGE_CHARS]
        return {"model": model, "works": "no", "reason": reason, "limits": quota_limits(error)}
    except httpx.TimeoutException:
        reason = f"no answer within {GENERATE_TIMEOUT_SECONDS}s"
        return {"model": model, "works": "no", "reason": reason, "limits": {}}
    return {"model": model, "works": "yes", "reason": "", "limits": header_limits(response)}


def daily_limit(limits: dict[str, str]) -> int | None:
    """Return the smallest per-day limit among `limits`, or None if none is exposed."""
    daily = [
        int(value)
        for name, value in limits.items()
        if DAILY_NAME_PART in name.lower() and value.isdigit()
    ]
    return min(daily, default=None)


def run_free_tier_probe() -> None:
    """Probe every candidate, print the table, and stop if the model choice is wrong."""
    probes = []
    for model in FREE_TIER_CANDIDATES:
        print(f"probing {model} ...", flush=True)
        started = time.perf_counter()
        probe = probe_free_generation(model)
        api_calls["probe"] += 1
        elapsed = time.perf_counter() - started
        print(f"  works={probe['works']} after {elapsed:.1f}s {probe['reason']}", flush=True)
        probes.append(probe)
    probe_table = pd.DataFrame(probes).set_index("model")
    probe_table["limits"] = probe_table["limits"].map(lambda limits: limits or "not exposed")
    print(probe_table.to_string())

    free_models = [probe["model"] for probe in probes if probe["works"] == "yes"]
    if not free_models:
        raise RuntimeError("No candidate model generates for free on this key")
    if COMPRESSOR_MODEL != free_models[0]:
        raise RuntimeError(
            f"Set COMPRESSOR_MODEL = {free_models[0]!r} in cell 3: "
            "it is the strongest model that generates for free"
        )
    for model in {QUESTION_MODEL, ANSWERER_MODEL, JUDGE_MODEL}:
        if model not in free_models:
            raise RuntimeError(f"{model} does not generate for free on this key")

    compressor_probe = next(probe for probe in probes if probe["model"] == COMPRESSOR_MODEL)
    compressor_daily = daily_limit(compressor_probe["limits"])
    if compressor_daily is None:
        print(
            f"\n{COMPRESSOR_MODEL}: daily limit not exposed by the API. Check it in AI Studio; "
            f"compression needs {N_COMPRESSION_CALLS} calls."
        )
    elif compressor_daily < N_COMPRESSION_CALLS:
        raise RuntimeError(
            f"{COMPRESSOR_MODEL} allows {compressor_daily} requests/day, "
            f"compression needs {N_COMPRESSION_CALLS}"
        )
    else:
        print(f"\n{COMPRESSOR_MODEL}: {compressor_daily} requests/day >= {N_COMPRESSION_CALLS}")


if RUN_FREE_TIER_PROBE:
    run_free_tier_probe()
else:
    print("Probe skipped (RUN_FREE_TIER_PROBE = False); free-tier limits come from AI Studio.")

## 5. Load data

Take the first 20 paragraphs of notebook 01's FLORES+ dev sample (50 paragraphs, seed 42).

In [ ]:
pairs = load_aligned_pairs()
paragraphs = build_paragraphs(pairs, N_SOURCE_PARAGRAPHS, SEED).head(N_PARAGRAPHS)
he_by_pair = paragraphs.set_index("pair_id")["he"]

print(f"{len(paragraphs)} paragraphs (first {N_PARAGRAPHS} of notebook 01's sample)")
print(f"Avg Hebrew words per paragraph: {paragraphs['he'].str.split().str.len().mean():.0f}")
print(paragraphs[["pair_id", "topic", "n_sentences"]].to_string(index=False))

## 6. Token counters

`o200k_base` locally, plus the free Gemini `count_tokens` of `gemini-3.8-flash` and
`gemini-4-argon` (different tokenizers in notebook 01), each with its fixed overhead subtracted.

In [ ]:
O200K_MODEL = "gpt-4o"


def cached_gemini_count(model: str, text: str) -> int:
    """Raw count_tokens of `model` for `text` (overhead included), cached when possible."""
    return cached_call(
        "count_tokens",
        {"model": model, "text": text},
        partial(raw_gemini_count, gemini_client, model, text),
        COUNT_PAUSE_SECONDS,
    )


o200k_counter = make_openai_counter(O200K_MODEL)
print(f"{O200K_MODEL}: tokenizer {o200k_counter.tokenizer_id}")
TOKEN_COUNTERS = {"o200k": o200k_counter}
for name, model in GEMINI_TOKEN_COUNT_MODELS.items():
    raw_count = partial(cached_gemini_count, model)
    TOKEN_COUNTERS[name] = make_gemini_counter(
        model, measure_overhead(model, raw_count), raw_count
    )

## 7. Compression by deletion

Ask the oracle for each target, then check in code that the kept words are an in-order
subsequence of the original words (punctuation ignored). Violations are logged and discarded.
A compression that fails with a 503, a 504 or a timeout is logged and skipped; it is not cached,
so the next run retries it.

In [ ]:
COMPRESS_PROMPT = """You shorten Hebrew text by deleting words. Nothing else is allowed.

Rules:
- Only delete whole words. Never reword, add, translate, or reorder words.
- Copy every word you keep exactly as it appears in the original, letter for letter, \
including prefixes such as ו, ה, ב, ל, מ, ש, כ.
- Keep all names, numbers, dates, and negations.
- Keep the main meaning and the key facts.

The original has {n_words} words. Delete about {percent}% of them, \
so that about {target_words} words remain.

Original text:
{text}"""
COMPRESSION_COLUMNS = [
    "pair_id",
    "target",
    "compressed",
    "valid",
    "original_words",
    "compressed_words",
]
SERVICE_UNAVAILABLE_CODE = 503
SKIPPED_ERROR_CODES = frozenset({SERVICE_UNAVAILABLE_CODE, GATEWAY_TIMEOUT_CODE})


def normalize_word(word: str) -> str:
    """Drop punctuation characters and casefold Latin letters."""
    return "".join(
        char for char in word if not unicodedata.category(char).startswith("P")
    ).casefold()


def to_words(text: str) -> list[str]:
    """Split `text` into normalized words; dashes, including the Hebrew maqaf, split words."""
    spaced = "".join(
        " " if unicodedata.category(char) == "Pd" else char
        for char in unicodedata.normalize("NFC", text)
    )
    return [word for word in map(normalize_word, spaced.split()) if word]


def first_violation(kept: list[str], original: list[str]) -> str | None:
    """Return the first kept word that breaks the in-order subsequence, or None if valid."""
    position = 0
    for word in kept:
        try:
            position = original.index(word, position) + 1
        except ValueError:
            return word
    return None


def compress(text: str, fraction: float) -> str:
    """Ask the oracle to delete about `fraction` of the words of `text`."""
    n_words = len(text.split())
    prompt = COMPRESS_PROMPT.format(
        n_words=n_words,
        percent=round(100 * fraction),
        target_words=round(n_words * (1 - fraction)),
        text=text,
    )
    return generate_json("compress", COMPRESSOR_MODEL, prompt, Compression).compressed


def try_compress(pair_id: str, target: str, text: str, fraction: float) -> str | None:
    """Return the compression, or None after logging a 503, 504 or timeout (retried next run)."""
    try:
        return compress(text, fraction)
    except TimeoutError as error:
        reason = str(error)
    except errors.APIError as error:
        if error.code not in SKIPPED_ERROR_CODES:
            raise
        reason = f"{error.code} {error.status}"
    print(f"failed {pair_id} {target}: {reason}")
    time.sleep(generate_pause_seconds(COMPRESSOR_MODEL))
    return None


rows = []
failed = []
for _, paragraph in paragraphs.iterrows():
    original_words = to_words(paragraph["he"])
    for target, fraction in TARGETS.items():
        compressed = try_compress(paragraph["pair_id"], target, paragraph["he"], fraction)
        if compressed is None:
            failed.append((paragraph["pair_id"], target))
            continue
        violation = first_violation(to_words(compressed), original_words)
        if violation is not None:
            print(f"violation {paragraph['pair_id']} {target}: {violation!r} breaks the order")
        rows.append(
            {
                "pair_id": paragraph["pair_id"],
                "target": target,
                "compressed": compressed,
                "valid": violation is None,
                "original_words": len(paragraph["he"].split()),
                "compressed_words": len(compressed.split()),
            }
        )
compressions = pd.DataFrame(rows, columns=COMPRESSION_COLUMNS)
is_valid = compressions["valid"].astype(bool)
compressions["word_reduction_pct"] = (
    100 * (1 - compressions["compressed_words"] / compressions["original_words"])
).where(is_valid)

violation_pct = 100 * (1 - compressions.groupby("target")["valid"].mean())
print(f"violation rate (%): {violation_pct.round(1).to_dict()}")
print(
    f"done {len(rows)}/{N_COMPRESSION_CALLS}, failed {len(failed)}, "
    f"key in use: key {key_index + 1}"
)

## 8. Token savings

Count original and compressed Hebrew with each tokenizer, reported separately
(valid compressions only).

In [ ]:
for name, counter in TOKEN_COUNTERS.items():
    original_tokens = he_by_pair.map(counter.count)
    compressions[f"{name}_original"] = compressions["pair_id"].map(original_tokens)
    compressions.loc[is_valid, f"{name}_compressed"] = compressions.loc[
        is_valid, "compressed"
    ].map(counter.count)
    compressions[f"{name}_savings_pct"] = 100 * (
        1 - compressions[f"{name}_compressed"] / compressions[f"{name}_original"]
    )

savings_table = compressions.pivot(
    index="pair_id",
    columns="target",
    values=["word_reduction_pct"] + [f"{name}_savings_pct" for name in TOKEN_COUNTERS],
)
print(savings_table.round(1).to_string())

## 9. Quality check

Write 3 factual questions per original paragraph, answer them from the original and from each
valid compression (one call per text), then judge each compressed answer against the original one.

In [ ]:
QUESTIONS_PROMPT = """Write exactly {n} factual questions in Hebrew about the Hebrew paragraph below.
- Each question must be answerable from the paragraph alone, with a short answer \
(a name, number, date, place, or short phrase).
- Each question must ask about a different fact.

Paragraph:
{text}"""

ANSWERS_PROMPT = """Answer the {n} questions below using ONLY the Hebrew text that follows. \
Do not use outside knowledge.
- Answer each question briefly, in Hebrew.
- If the text does not contain the answer, answer exactly: {not_in_text}
- Return exactly {n} answers, in the order of the questions.

Questions:
{questions}

Text:
{text}"""

JUDGE_PROMPT = """For each question below, compare two answers:
- Reference answer: written from the full text.
- Candidate answer: written from a shortened version of the text.

Verdicts:
- match: the candidate states the same facts as the reference.
- partial: the candidate is correct but less complete or less specific than the reference.
- mismatch: the candidate is wrong, contradicts the reference, or says the information is missing.

Return exactly {n} verdicts, in order, each with a one-line reason in English.

{items}"""


def numbered(lines: list[str]) -> str:
    """Join `lines` as a 1-based numbered list."""
    return "\n".join(f"{index}. {line}" for index, line in enumerate(lines, start=1))


def ask_questions(text: str) -> list[str]:
    """Return the oracle's factual questions about `text`."""
    prompt = QUESTIONS_PROMPT.format(n=N_QUESTIONS, text=text)
    return generate_json("questions", QUESTION_MODEL, prompt, Questions).questions


def answer(text: str, questions: list[str]) -> list[str]:
    """Answer all `questions` from `text` alone, in one call."""
    prompt = ANSWERS_PROMPT.format(
        n=N_QUESTIONS,
        not_in_text=NOT_IN_TEXT,
        questions=numbered(questions),
        text=text,
    )
    return generate_json("answers", ANSWERER_MODEL, prompt, Answers).answers


def judge(questions: list[str], reference: list[str], candidate: list[str]) -> list[Verdict]:
    """Judge each candidate answer against the reference answer, in one call."""
    items = "\n\n".join(
        f"Question {index}: {question}\n"
        f"Reference answer: {reference_answer}\n"
        f"Candidate answer: {candidate_answer}"
        for index, (question, reference_answer, candidate_answer) in enumerate(
            zip(questions, reference, candidate), start=1
        )
    )
    prompt = JUDGE_PROMPT.format(n=N_QUESTIONS, items=items)
    return generate_json("judge", JUDGE_MODEL, prompt, Judgement).verdicts


questions_by_pair: dict[str, list[str]] = {}
reference_by_pair: dict[str, list[str]] = {}
for pair_id, text in he_by_pair.items():
    questions_by_pair[pair_id] = ask_questions(text)
    reference_by_pair[pair_id] = answer(text, questions_by_pair[pair_id])

judged_rows = []
for _, row in compressions[is_valid].iterrows():
    questions = questions_by_pair[row["pair_id"]]
    reference = reference_by_pair[row["pair_id"]]
    candidate = answer(row["compressed"], questions)
    verdicts = judge(questions, reference, candidate)
    for index, (question, reference_answer, candidate_answer, verdict) in enumerate(
        zip(questions, reference, candidate, verdicts)
    ):
        judged_rows.append(
            {
                "pair_id": row["pair_id"],
                "target": row["target"],
                "question_index": index,
                "question": question,
                "original_answer": reference_answer,
                "compressed_answer": candidate_answer,
                "verdict": verdict.verdict,
                "reason": verdict.reason,
            }
        )
judgements = pd.DataFrame(judged_rows)

verdict_counts = (
    judgements.groupby(["pair_id", "target"])["verdict"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=list(VERDICTS), fill_value=0)
    .add_prefix("n_")
    .reset_index()
)
compressions = compressions.drop(
    columns=[f"n_{verdict}" for verdict in VERDICTS], errors="ignore"
).merge(verdict_counts, on=["pair_id", "target"], how="left")
print(f"judged {len(judgements)} questions")
print(judgements.groupby("target")["verdict"].value_counts().unstack(fill_value=0))

## 10. Results

One row per target, saved per paragraph (counts and scores only, no text) to `results/`.

In [ ]:
CSV_COLUMNS = (
    ["pair_id", "target", "valid", "original_words", "compressed_words", "word_reduction_pct"]
    + [
        f"{name}_{measure}"
        for name in TOKEN_COUNTERS
        for measure in ("original", "compressed", "savings_pct")
    ]
    + [f"n_{verdict}" for verdict in VERDICTS]
)

verdict_pct = (
    judgements.groupby("target")["verdict"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=list(VERDICTS), fill_value=0)
    .mul(100)
    .add_suffix("_pct")
)
savings_columns = [f"{name}_savings_pct" for name in TOKEN_COUNTERS]
summary = (
    compressions.groupby("target")
    .agg(
        paragraphs=("pair_id", "size"),
        word_reduction_pct=("word_reduction_pct", "mean"),
        **{column: (column, "mean") for column in savings_columns},
    )
    .join(verdict_pct)
    .assign(violation_pct=violation_pct)
)
print(summary.round(1).to_string())

compressions[CSV_COLUMNS].to_csv(RESULTS_CSV, index=False)
print(f"\n{len(compressions)} rows -> {RESULTS_CSV}")
print(f"API calls this session (cache misses): {dict(api_calls)}, total {api_calls.total()}")

## 11. Chart

Token savings vs. match rate, one point per target for each tokenizer.

In [ ]:
TOKENIZER_COLORS = {
    "o200k": "tab:green",
    "gemini_3_8_flash": "tab:blue",
    "gemini_4_argon": "tab:purple",
}

fig, ax = plt.subplots(figsize=(8, 6))
for name, counter in TOKEN_COUNTERS.items():
    column = f"{name}_savings_pct"
    label = counter.tokenizer_id or counter.model
    color = TOKENIZER_COLORS.get(name, "tab:gray")
    ax.scatter(summary[column], summary["match_pct"], s=80, color=color, label=label)
    for target, x, y in zip(summary.index, summary[column], summary["match_pct"]):
        ax.annotate(f"{target} target", (x, y), xytext=(6, 6), textcoords="offset points")

ax.set_xlim(0, 100)
ax.set_ylim(0, 105)
ax.set_xlabel("Mean token savings (%)", fontsize=12)
ax.set_ylabel("Answers matching the original (%)", fontsize=12)
ax.set_title(
    f"Deletion-only compression by {COMPRESSOR_MODEL} ({len(paragraphs)} FLORES+ paragraphs)",
    fontsize=13,
)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_PNG, dpi=200)
plt.show()

## 12. Example mismatches

Three mismatched answers for manual review.

In [ ]:
N_EXAMPLES = 3

mismatches = judgements[judgements["verdict"] == "mismatch"].head(N_EXAMPLES)
if mismatches.empty:
    print("No mismatches.")
for _, example in mismatches.iterrows():
    print(f"{example['pair_id']} ({example['target']} target)")
    print(f"  Q: {example['question']}")
    print(f"  original:   {example['original_answer']}")
    print(f"  compressed: {example['compressed_answer']}")
    print(f"  judge: {example['reason']}\n")

## 13. Findings